# Quant — Full NASDAQ Data Refresh (Colab)

이 노트북은 `cache/`를 비운 뒤 Massive.com으로 **전체 NASDAQ 유니버스 → 가격 → 재무**를 순서대로 갱신합니다.

**주의**
- 전체 유니버스(~4,000+ active + delisted) profile/가격/재무 다운로드는 **수 시간** 걸릴 수 있습니다.
- Colab 세션 끊김 방지를 위해 가능하면 **Google Drive 저장** 셀을 사용하세요.
- Massive API 키와 Financials 권한이 필요합니다.

## 0) 설정

Colab 왼쪽 🔑 **Secrets**에 `MASSIVE_API_KEY`를 등록하거나, 아래 셀에서 직접 입력하세요.

In [ ]:
# ====== USER SETTINGS ======
REPO_URL = "https://github.com/ygkhxdbhs5-hash/Quant.git"
BRANCH = "cursor/full-nasdaq-universe-cb1c"  # full universe (no universe_limit)
REPO_DIR = "/content/Quant"

# True면 /content/drive/MyDrive/Quant_data 에 data/ + cache/ 백업
SAVE_TO_DRIVE = True
DRIVE_DATA_DIR = "/content/drive/MyDrive/Quant_data"

# 단계 선택 (전체 갱신이면 모두 True)
RUN_UNIVERSE = True
RUN_PRICES = True
RUN_FUNDAMENTALS = True
CLEAR_CACHE = True

## 1) Clone repo + install deps

In [ ]:
import os
import sys
from pathlib import Path

%cd /content

if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!pip -q install -r requirements.txt

sys.path.insert(0, str(Path.cwd()))
print("cwd:", Path.cwd())
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

## 2) API key + (optional) Google Drive

In [ ]:
import os
from pathlib import Path

# 1) Colab Secrets 우선
api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("MASSIVE_API_KEY") or ""
except Exception:
    pass

# 2) 환경변수
if not api_key:
    api_key = os.environ.get("MASSIVE_API_KEY") or os.environ.get("POLYGON_API_KEY") or ""

# 3) 직접 입력 (Secrets 미사용 시)
if not api_key:
    import getpass
    api_key = getpass.getpass("MASSIVE_API_KEY: ").strip()

if not api_key:
    raise RuntimeError("MASSIVE_API_KEY is required")

os.environ["MASSIVE_API_KEY"] = api_key
os.environ["PYTHONUNBUFFERED"] = "1"
print("API key set:", api_key[:4] + "…" + api_key[-4:])

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_DATA_DIR).mkdir(parents=True, exist_ok=True)
    print("Drive data dir:", DRIVE_DATA_DIR)

## 3) 캐시 삭제

이전 소규모(예: 30종목) HTTP 캐시가 남지 않도록 `cache/`를 비웁니다.

In [ ]:
import shutil
from pathlib import Path

cache_dir = Path("cache")
cache_dir.mkdir(exist_ok=True)

if CLEAR_CACHE:
    removed = 0
    for p in cache_dir.rglob("*"):
        if p.is_file() and p.name != ".gitkeep":
            p.unlink(missing_ok=True)
            removed += 1
    for p in sorted(cache_dir.rglob("*"), reverse=True):
        if p.is_dir() and p != cache_dir:
            try:
                p.rmdir()
            except OSError:
                pass
    (cache_dir / "massive_cache").mkdir(parents=True, exist_ok=True)
    (cache_dir / ".gitkeep").touch()
    print(f"Cleared cache files: {removed}")
else:
    print("CLEAR_CACHE=False — skipped")

print("cache contents:", list(cache_dir.iterdir()))

## 4) 전체 데이터 갱신 (순서 고정)

1. `download_universe` — 전체 NASDAQ (`tickers == all_tickers`, limit 무시)
2. `download_prices`
3. `download_fundamentals`

중간에 끊기면 **같은 셀을 다시 실행**하세요. HTTP 응답은 `cache/massive_cache`에 쌓이므로 이어받기됩니다.
(방금 캐시를 지웠다면 처음부터입니다.)

In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

CFG = ["--config", "config/config.yaml"]

def run_step(title: str, module: str) -> None:
    print("\n" + "=" * 72)
    print(f"[{datetime.now().isoformat(timespec='seconds')}] START {title}")
    print("=" * 72)
    env = dict(**os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.run(
        [sys.executable, "-u", "-m", module, *CFG],
        cwd=str(Path.cwd()),
        env=env,
    )
    print(f"[{datetime.now().isoformat(timespec='seconds')}] END {title} exit={proc.returncode}")
    if proc.returncode != 0:
        raise RuntimeError(f"{module} failed with exit {proc.returncode}")

if RUN_UNIVERSE:
    run_step("1/3 universe", "downloader.download_universe")
else:
    print("skip universe")

if RUN_PRICES:
    run_step("2/3 prices", "downloader.download_prices")
else:
    print("skip prices")

if RUN_FUNDAMENTALS:
    run_step("3/3 fundamentals", "downloader.download_fundamentals")
else:
    print("skip fundamentals")

print("\n✅ Data refresh pipeline finished")

## 5) 검증 — 유니버스가 전체인지 확인

In [ ]:
import pickle
from pathlib import Path

uni_path = Path("data/metadata/universe.pkl")
px_path = Path("data/prices/panels.pkl")
fund_path = Path("data/fundamentals/pit_history.pkl")

assert uni_path.exists(), f"missing {uni_path}"
uni = pickle.load(open(uni_path, "rb"))
tickers = uni.get("tickers") or []
all_tickers = uni.get("all_tickers") or []
profiles = uni.get("profile_meta") or {}

print(f"all_tickers     : {len(all_tickers)}")
print(f"tickers         : {len(tickers)}")
print(f"profile_meta    : {len(profiles)}")
print(f"tickers==all?   : {tickers == all_tickers}")

if len(tickers) < 1000:
    print("⚠️ tickers < 1000 — 전체 유니버스가 아닐 수 있습니다. 브랜치/캐시를 확인하세요.")
else:
    print("✅ universe size looks like a full NASDAQ pull")

if px_path.exists():
    panels = pickle.load(open(px_path, "rb"))
    close_m = panels["close_m"]
    print(f"prices close_m  : {close_m.shape}  ({close_m.index.min().date()} → {close_m.index.max().date()})")
else:
    print("prices: not found yet")

if fund_path.exists():
    funds = pickle.load(open(fund_path, "rb"))
    print(f"fundamentals    : {len(funds)} symbols with PIT history")
else:
    print("fundamentals: not found yet")

## 6) (권장) Drive에 결과 백업

In [ ]:
import shutil
from pathlib import Path

if not SAVE_TO_DRIVE:
    print("SAVE_TO_DRIVE=False — skip")
else:
    src_data = Path("data")
    src_cache = Path("cache")
    dst = Path(DRIVE_DATA_DIR)
    dst.mkdir(parents=True, exist_ok=True)

    for name, src in [("data", src_data), ("cache", src_cache)]:
        target = dst / name
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(src, target)
        print(f"copied {src} -> {target}")

    print("Drive backup done:", dst)
    !du -sh {DRIVE_DATA_DIR}/data {DRIVE_DATA_DIR}/cache 2>/dev/null || true

## 7) (선택) 백테스트 실행

In [ ]:
# 데이터 갱신 후에만 실행
RUN_BACKTEST = False

if RUN_BACKTEST:
    !python -u run_backtest.py --config config/config.yaml
else:
    print("Set RUN_BACKTEST=True to run the engine")

### 참고: Drive에 저장된 데이터로 다음 세션 복원

새 Colab 세션에서 다운로드를 건너뛰려면:

```python
from google.colab import drive
import shutil
from pathlib import Path
drive.mount('/content/drive')
# repo clone 후:
for name in ['data', 'cache']:
    src = Path('/content/drive/MyDrive/Quant_data') / name
    dst = Path('/content/Quant') / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
```